In [12]:
import pandas as pd
from pathlib import Path

# Util for normalizing subcategory name
def _fix_personal_episodic_name(tag):
    if tag == "Prior Knowledge: Personal /Episodic":
        return "Prior Knowledge: Personal / Episodic"
    return tag

# Paths relative to this notebook
HERE = Path.cwd()
PIPELINE = HERE.parent
COMBINED_DATA_DIR = PIPELINE / "combined_data"
DATAFRAMES_DIR = PIPELINE / "dataframes"

# Tag cleanup from subcategory_dependency_chord.py
_TAG_CLEANUP = {
    "L3: Trend and pattern analysis": "Visual Observation: Cross-point Pattern Recognition",
    "L1: Elemental and encoded properties": "Visual Observation: Chart Structure & Text",
    "L2: Statistical concepts and relations": "Visual Observation: Data Point Extraction",
    "VO1: Chart Structure, Layout & Text": "Visual Observation: Chart Structure & Text",
    "VO2: Data Point Reading": "Visual Observation: Data Point Extraction",
    "VO3: Comparisons, Trends & Patterns": "Visual Observation: Cross-point Pattern Recognition",
    "Background knowledge": "Prior Knowledge: Background",
    "Personal/episodic retrieval": "Prior Knowledge: Personal / Episodic",
    "Evaluative / affective judgment": "Evaluative: Reactive",
    "Explanatory inference": "Inference: Explanatory",
    "Predictive / counterfactual inference": "Inference: Predictive / Hypothetical",
    "Information need / curiosity": "Curiosity",
    "Meta /Paratext": "Meta / Paratext",
    "Meta / paratext": "Meta / Paratext",
}

EXCLUDED_SUBCATS = {"Meta / Paratext", "Uncategorizable"}

def _norm_tag(tag):
    import numpy as np
    if tag is None or (isinstance(tag, float) and pd.isna(tag)):
        return None
    s = str(tag).strip()
    s = _fix_personal_episodic_name(s)
    return _TAG_CLEANUP.get(s, s)

def edges_from_record(record):
    graph = record.get("dependency_graph") or []
    if not graph:
        return []
    nodes = {n["id"]: n for n in graph if "id" in n}
    article_id = str(record.get("article_id", ""))
    comment_index = record.get("comment_index")

    rows = []
    for node in graph:
        cid = node.get("id")
        if cid is None:
            continue
        child_tag = _norm_tag(node.get("comment_tag"))
        for dep in node.get("depends_on") or []:
            if not isinstance(dep, dict):
                continue
            pid = dep.get("id")
            parent = nodes.get(pid)
            if parent is None:
                continue
            parent_tag = _norm_tag(parent.get("comment_tag"))
            edge_type = dep.get("edge_type")
            rows.append(
                {
                    "article_id": article_id,
                    "comment_index": comment_index,
                    "parent_id": pid,
                    "child_id": cid,
                    "parent_subcategory": parent_tag,
                    "child_subcategory": child_tag,
                    "edge_category": edge_type if edge_type is not None else "",
                }
            )
    return rows

def load_combined_edges(combined_dir):
    import json
    import numpy as np
    paths = sorted(combined_dir.glob("*.json"))
    if not paths:
        raise FileNotFoundError(f"No JSON files under {combined_dir}")

    all_rows = []
    for p in paths:
        with p.open(encoding="utf-8") as f:
            data = json.load(f)
        if not isinstance(data, list):
            continue
        for record in data:
            if isinstance(record, dict):
                all_rows.extend(edges_from_record(record))

    df = pd.DataFrame(all_rows)
    if df.empty:
        return df

    mask = (
        df["parent_subcategory"].notna()
        & df["child_subcategory"].notna()
        & ~df["parent_subcategory"].isin(EXCLUDED_SUBCATS)
        & ~df["child_subcategory"].isin(EXCLUDED_SUBCATS)
    )
    return df.loc[mask].reset_index(drop=True)

# Actually load the dataframe
df = load_combined_edges(COMBINED_DATA_DIR)
df.head()

,article_id,comment_index,parent_id,child_id,parent_subcategory,child_subcategory,edge_category
0,1,1,0,2,Visual Observation: Data Point Extraction,Visual Observation: Cross-point Pattern Recogn...,Elaborative
1,1,1,1,2,Visual Observation: Data Point Extraction,Visual Observation: Cross-point Pattern Recogn...,Elaborative
2,1,1,2,3,Visual Observation: Cross-point Pattern Recogn...,Curiosity,Evaluative
3,1,1,2,4,Visual Observation: Cross-point Pattern Recogn...,Prior Knowledge: Personal / Episodic,Inferential
4,1,1,0,5,Visual Observation: Data Point Extraction,Visual Observation: Data Point Extraction,Elaborative


In [21]:
import altair as alt
import numpy as np

# -- Remove "Unknown" and color subcategories according to parent node as in 1_across_plot.ipynb --
# You may want to import the relevant lists if available, but for illustration,
# let's filter out subcategories containing "Unknown" or "Color" in the parent_subcategory

# Explicit "bad" subcats based on 1_across_plot.ipynb / category logic
# Substitute these for yours as appropriate from your project context
EXCLUDED_PARENTS = [
    "Unknown",
    "Meta / Paratext",
    "Meta/Paratext",
    "Color",  # Remove parents related to color as needed
]
# Also drop *any* parent containing those as substring for potential variants
def _parent_is_good(x):
    bads = ["unknown", "meta", "paratext", "color"]
    if not isinstance(x, str):
        return False
    return not any(bad in x.lower() for bad in bads)

filtered_df = df[df['parent_subcategory'].apply(_parent_is_good)].copy()

# Create a co-occurrence matrix: counts of edges for each parent_subcategory -> child_subcategory
cooccurrence = (
    filtered_df.groupby(["parent_subcategory", "child_subcategory"])
      .size()
      .reset_index(name="count")
)

# Add a "count_display" column: rounded to the nearest 0.1 thousands (40,320 -> 40.3)
def count_to_k(x):
    if x < 1000:
        return str(x)
    return f"{x/1000:.1f}k"

cooccurrence["count_display"] = cooccurrence["count"].apply(count_to_k)

# For a consistent axis order, get the sorted unique subcategories after filtering:
subcats = sorted(
    set(cooccurrence["parent_subcategory"]).union(cooccurrence["child_subcategory"])
)

base = alt.Chart(cooccurrence)

rects = (
    base.mark_rect()
    .encode(
        x=alt.X(
            "parent_subcategory:N", 
            sort=subcats, 
            title="Parent Subcategory"
        ),
        y=alt.Y(
            "child_subcategory:N", 
            sort=subcats, 
            title="Child Subcategory"
        ),
        color=alt.Color(
            "count:Q", 
            scale=alt.Scale(scheme="blues", type="sqrt"), 
            legend=alt.Legend(title="Edge Count"),
        ),
        tooltip=[
            alt.Tooltip("parent_subcategory:N", title="Parent"),
            alt.Tooltip("child_subcategory:N", title="Child"),
            alt.Tooltip("count:Q", title="Edge Count"),
            alt.Tooltip("count_display:N", title="Rounded count"),
        ],
    )
)

text = (
    base.mark_text(baseline='middle', fontSize=11, color='black')
    .encode(
        x=alt.X(
            "parent_subcategory:N",
            sort=subcats
        ),
        y=alt.Y(
            "child_subcategory:N",
            sort=subcats
        ),
        text=alt.Text("count_display:N")
    )
)

chart = (
    (rects + text)
    .properties(
        width=500,
        height=500,
        title="Dependency Graph Edge Co-occurrence Matrix (parent vs child subcategory)"
    )
)

chart.display()

alt.LayerChart(...)

In [10]:
# Per-subcategory node roles across all comments (within-comment dependency graphs only).
# Leaf / sink: node has no outgoing edges to other nodes in the same graph (nothing depends on it).
# Root: node has no incoming edges from other nodes in the same graph (empty or invalid depends_on).

import json
from collections import defaultdict

def iter_graph_records(combined_dir):
    for p in sorted(combined_dir.glob("*.json")):
        with p.open(encoding="utf-8") as f:
            data = json.load(f)
        if not isinstance(data, list):
            continue
        for record in data:
            if isinstance(record, dict) and record.get("dependency_graph"):
                yield record

# Util for normalizing subcategory name
def _fix_personal_episodic_name(tag):
    if tag == "Prior Knowledge: Personal /Episodic":
        return "Prior Knowledge: Personal / Episodic"
    return tag

leaf_counts = defaultdict(int)
root_counts = defaultdict(int)
total_by_subcat = defaultdict(int)

for record in iter_graph_records(COMBINED_DATA_DIR):
    graph = record.get("dependency_graph") or []
    nodes = {n["id"]: n for n in graph if "id" in n}
    children = {nid: set() for nid in nodes}
    for n in graph:
        cid = n.get("id")
        if cid is None:
            continue
        for dep in n.get("depends_on") or []:
            if not isinstance(dep, dict):
                continue
            pid = dep.get("id")
            if pid in nodes:
                children.setdefault(pid, set()).add(cid)

    for nid, node in nodes.items():
        tag = _norm_tag(node.get("comment_tag"))
        if tag is None:
            continue
        tag = _fix_personal_episodic_name(tag)
        total_by_subcat[tag] += 1
        if not children.get(nid):
            leaf_counts[tag] += 1
        deps = node.get("depends_on") or []
        has_parent = any(isinstance(d, dict) and d.get("id") in nodes for d in deps)
        if not has_parent:
            root_counts[tag] += 1

leaf_rows = [
    {
        "subcategory": k,
        "leaf_node_count": leaf_counts[k],
        "total_nodes": total_by_subcat[k],
        "frac_leaf": leaf_counts[k] / total_by_subcat[k],
    }
    for k in sorted(total_by_subcat.keys())
]
root_rows = [
    {
        "subcategory": k,
        "root_node_count": root_counts[k],
        "total_nodes": total_by_subcat[k],
        "frac_root": root_counts[k] / total_by_subcat[k],
    }
    for k in sorted(total_by_subcat.keys())
]

leaf_by_subcat = pd.DataFrame(leaf_rows).sort_values(
    ["leaf_node_count", "subcategory"], ascending=[False, True]
)
root_by_subcat = pd.DataFrame(root_rows).sort_values(
    ["root_node_count", "subcategory"], ascending=[False, True]
)

display(leaf_by_subcat.sort_values(by="leaf_node_count"))
root_by_subcat.sort_values(by="root_node_count")

,subcategory,leaf_node_count,total_nodes,frac_leaf
8,Uncategorizable,4553,9172,0.496402
1,Evaluative: Prescriptive,4932,8179,0.603008
12,unknown,6380,12395,0.514724
2,Evaluative: Reactive,7904,12480,0.633333
11,Visual Observation: Data Point Extraction,12343,28997,0.425665
9,Visual Observation: Chart Structure & Text,14038,35745,0.392726
3,Inference: Explanatory,17826,28341,0.628983
4,Inference: Predictive / Hypothetical,20236,32289,0.626715
7,Prior Knowledge: Personal / Episodic,21598,46425,0.465223
6,Prior Knowledge: Background,25025,50428,0.496252


,subcategory,root_node_count,total_nodes,frac_root
1,Evaluative: Prescriptive,424,8179,0.051840
3,Inference: Explanatory,934,28341,0.032956
2,Evaluative: Reactive,1312,12480,0.105128
4,Inference: Predictive / Hypothetical,1561,32289,0.048345
12,unknown,3000,12395,0.242033
8,Uncategorizable,4494,9172,0.489969
0,Curiosity,4855,69343,0.070014
7,Prior Knowledge: Personal / Episodic,7907,46425,0.170318
11,Visual Observation: Data Point Extraction,9154,28997,0.315688
6,Prior Knowledge: Background,10339,50428,0.205025


If Visual Observation is a root node and Curiosit is a leaf node, how many pairs in between?

In [22]:
# Structural root tagged VO3 (normalized) → structural leaf tagged Curiosity.
# Root / leaf match the definitions above: no incoming vs no outgoing edges in the comment graph.
# Edge direction: parent → child when the child node depends_on the parent.

import json
import networkx as nx

ROOT_SUBCAT = _TAG_CLEANUP["VO3: Comparisons, Trends & Patterns"]
LEAF_SUBCAT = "Curiosity"


def collect_vo3_root_curiosity_leaf_chains(combined_dir, max_paths_per_record=None):
    rows = []
    for p in sorted(combined_dir.glob("*.json")):
        with p.open(encoding="utf-8") as f:
            data = json.load(f)
        if not isinstance(data, list):
            continue
        for record in data:
            graph = record.get("dependency_graph") if isinstance(record, dict) else None
            if not graph:
                continue
            nodes = {n["id"]: n for n in graph if "id" in n}
            if not nodes:
                continue
            G = nx.DiGraph()
            G.add_nodes_from(nodes)
            out_neighbors = {nid: set() for nid in nodes}
            in_neighbors = {nid: set() for nid in nodes}
            for n in graph:
                cid = n.get("id")
                if cid is None:
                    continue
                for dep in n.get("depends_on") or []:
                    if not isinstance(dep, dict):
                        continue
                    pid = dep.get("id")
                    if pid not in nodes:
                        continue
                    G.add_edge(pid, cid)
                    out_neighbors[pid].add(cid)
                    in_neighbors[cid].add(pid)

            roots = [nid for nid in nodes if not in_neighbors[nid]]
            leaves = [nid for nid in nodes if not out_neighbors[nid]]

            def tag_of(nid):
                return _norm_tag(nodes[nid].get("comment_tag"))

            starts = [nid for nid in roots if tag_of(nid) == ROOT_SUBCAT]
            ends = [nid for nid in leaves if tag_of(nid) == LEAF_SUBCAT]

            article_id = str(record.get("article_id", ""))
            comment_index = record.get("comment_index")

            n_kept = 0
            for s in starts:
                for t in ends:
                    if s == t:
                        continue
                    for path in nx.all_simple_paths(G, s, t):
                        rows.append(
                            {
                                "article_id": article_id,
                                "comment_index": comment_index,
                                "path_ids": list(path),
                                "path_tags": tuple(tag_of(u) for u in path),
                                "path_len": len(path),
                            }
                        )
                        n_kept += 1
                        if (
                            max_paths_per_record is not None
                            and n_kept >= max_paths_per_record
                        ):
                            break
                    if (
                        max_paths_per_record is not None
                        and n_kept >= max_paths_per_record
                    ):
                        break
                if max_paths_per_record is not None and n_kept >= max_paths_per_record:
                    break

    return pd.DataFrame(rows)


# Set max_paths_per_record if a single comment graph explodes (rare).
vo3_to_curiosity_chains = collect_vo3_root_curiosity_leaf_chains(
    COMBINED_DATA_DIR, max_paths_per_record=None
)
print(f"Total chains (VO3 structural root → Curiosity structural leaf): {len(vo3_to_curiosity_chains)}")
if len(vo3_to_curiosity_chains):
    display(vo3_to_curiosity_chains.head(25))
    print("Top tag sequences by frequency:")
    display(
        vo3_to_curiosity_chains.groupby("path_tags", as_index=False)
        .size()
        .sort_values("size", ascending=False)
        .head(40)
    )


Total chains (VO3 structural root → Curiosity structural leaf): 35435


,article_id,comment_index,path_ids,path_tags,path_len
0,1,3,"[1, 3]",(Visual Observation: Cross-point Pattern Recog...,2
1,1,3,"[2, 4]",(Visual Observation: Cross-point Pattern Recog...,2
2,1,4,"[0, 1]",(Visual Observation: Cross-point Pattern Recog...,2
3,1,5,"[0, 1, 8]",(Visual Observation: Cross-point Pattern Recog...,3
4,1,5,"[0, 7, 8]",(Visual Observation: Cross-point Pattern Recog...,3
5,1,5,"[0, 7, 9]",(Visual Observation: Cross-point Pattern Recog...,3
6,1,6,"[0, 1, 5]",(Visual Observation: Cross-point Pattern Recog...,3
7,1,6,"[0, 5]",(Visual Observation: Cross-point Pattern Recog...,2
8,1,10,"[1, 7]",(Visual Observation: Cross-point Pattern Recog...,2
9,1,10,"[1, 10]",(Visual Observation: Cross-point Pattern Recog...,2


Top tag sequences by frequency:


,path_tags,size
0,(Visual Observation: Cross-point Pattern Recog...,19380
374,(Visual Observation: Cross-point Pattern Recog...,5339
1,(Visual Observation: Cross-point Pattern Recog...,2410
504,(Visual Observation: Cross-point Pattern Recog...,996
630,(Visual Observation: Cross-point Pattern Recog...,790
375,(Visual Observation: Cross-point Pattern Recog...,631
314,(Visual Observation: Cross-point Pattern Recog...,409
104,(Visual Observation: Cross-point Pattern Recog...,384
152,(Visual Observation: Cross-point Pattern Recog...,331
204,(Visual Observation: Cross-point Pattern Recog...,275


In [39]:
# Unpack each chain's subcategories at each path position into separate columns (root, child1, child2, etc.)
# and sort by how often the chain occurs

# First, get the maximum path length to know how many columns to create
max_chain_len = vo3_to_curiosity_chains["path_tags"].apply(len).max()

# Create new column names: 'node_0', 'node_1', ..., 'node_{max_chain_len-1}'
node_cols = [f"node_{i}" for i in range(max_chain_len)]

# Count frequency of each unique chain (based on path_tags)
chain_freq = vo3_to_curiosity_chains["path_tags"].value_counts()
chains_expanded = vo3_to_curiosity_chains.copy()
# Map frequency count to each record
chains_expanded["chain_count"] = chains_expanded["path_tags"].map(chain_freq)

# Build a new DataFrame by expanding the path_tags (tuples) into these columns
chains_expanded[node_cols] = pd.DataFrame(
    chains_expanded["path_tags"].apply(lambda tags: list(tags) + [None]*(max_chain_len-len(tags))).tolist(),
    index=chains_expanded.index
)

# Now sort by chain_count (frequency)
chains_expanded_sorted = chains_expanded.sort_values("chain_count", ascending=False)

# Drop article_id and comment_index
chains_expanded_sorted = chains_expanded_sorted.drop(columns=["article_id", "comment_index"])

# Show a preview for inspection
chains_expanded_sorted[node_cols + ["chain_count"]].drop_duplicates().head(20)

,node_0,node_1,node_2,node_3,node_4,node_5,node_6,node_7,node_8,node_9,node_10,node_11,node_12,node_13,node_14,chain_count
0,Visual Observation: Cross-point Pattern Recogn...,Curiosity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19380
25852,Visual Observation: Cross-point Pattern Recogn...,Visual Observation: Cross-point Pattern Recogn...,Curiosity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5339
34893,Visual Observation: Cross-point Pattern Recogn...,Curiosity,Curiosity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2410
9573,Visual Observation: Cross-point Pattern Recogn...,Visual Observation: Cross-point Pattern Recogn...,Visual Observation: Cross-point Pattern Recogn...,Curiosity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,996
30364,Visual Observation: Cross-point Pattern Recogn...,Visual Observation: Data Point Extraction,Curiosity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,790
29562,Visual Observation: Cross-point Pattern Recogn...,Visual Observation: Cross-point Pattern Recogn...,Curiosity,Curiosity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,631
23330,Visual Observation: Cross-point Pattern Recogn...,Visual Observation: Chart Structure & Text,Curiosity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,409
30422,Visual Observation: Cross-point Pattern Recogn...,Inference: Explanatory,Curiosity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,384
4324,Visual Observation: Cross-point Pattern Recogn...,Inference: Predictive / Hypothetical,Curiosity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,331
17953,Visual Observation: Cross-point Pattern Recogn...,Prior Knowledge: Background,Curiosity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,275


In [32]:
import altair as alt
import pandas as pd

# Prepare data for full chain (multi-node "Sankey-like") visualization of paths.
path_seq_counts = (
    vo3_to_curiosity_chains.groupby("path_tags")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

# Expand all transitions in all chains, with chain "id" for bundling
links = []
for idx, row in path_seq_counts.iterrows():
    tags = row["path_tags"]
    count = row["count"]
    for i in range(len(tags) - 1):
        source = tags[i]
        target = tags[i + 1]
        links.append({
            "chain_id": idx,
            "step": i,
            "source": source,
            "target": target,
            "count": count,
            "full_chain": str(tags),
        })

link_df = pd.DataFrame(links)

# Use a consistent and explicit tag order for y-pos (avoid JS errors from dict/set index)
seen_tags = pd.Series([tag for row in path_seq_counts["path_tags"] for tag in row]).unique()
tag_order = list(seen_tags)  # deterministic, no sets

tag_y = {tag: i for i, tag in enumerate(tag_order)}
link_df["source_y"] = link_df["source"].map(tag_y)
link_df["target_y"] = link_df["target"].map(tag_y)

# Assign "x" for each hop (step) in a chain
link_df["x"] = link_df["step"]
link_df["x2"] = link_df["step"] + 1

# A tag label reference for plotting
tag_labels = pd.DataFrame({
    "y": list(range(len(tag_order))),
    "tag": tag_order
})

highlight = alt.selection_point(fields=['full_chain'], on="mouseover", clear="mouseout")

lines = (
    alt.Chart(link_df)
    .mark_rule(opacity=0.28)
    .encode(
        x=alt.X("x:O", title="Chain Position (step)"),
        x2="x2:O",
        y=alt.Y("source_y:O", title="Tag", axis=None),
        y2="target_y:O",
        strokeWidth=alt.StrokeWidth("count:Q", scale=alt.Scale(range=[1, 16])),
        color=alt.Color("source:N", scale=alt.Scale(scheme="category20"), legend=None),
        tooltip=[alt.Tooltip("source:N"), alt.Tooltip("target:N"), alt.Tooltip("count:Q"), alt.Tooltip("full_chain:N")],
        detail="chain_id:N"
    )
    .add_params(highlight)
    .encode(
        opacity=alt.condition(highlight, alt.value(0.8), alt.value(0.18))
    )
)

# Node chart for each unique tag x position
node_rows = []
max_chain_length = max(len(x) for x in path_seq_counts["path_tags"])
for tag, y in tag_y.items():
    for step in range(max_chain_length):
        count = sum(row["path_tags"][step] == tag if step < len(row["path_tags"]) else False
                   for _, row in path_seq_counts.iterrows())
        if count > 0:
            node_rows.append({"tag": tag, "y": y, "x": step, "count": count})

node_df = pd.DataFrame(node_rows)

nodes = (
    alt.Chart(node_df)
    .mark_circle(size=400, opacity=0.9, color='black')
    .encode(
        x=alt.X("x:O"),
        y=alt.Y("y:O", title=None, axis=None),
        size=alt.Size("count:Q", scale=alt.Scale(range=[40, 600]), legend=None),
        tooltip=["tag:N", "count:Q"]
    )
)

row_labels = (
    alt.Chart(tag_labels)
    .mark_text(align="right", baseline="middle", dx=-8, fontSize=12)
    .encode(
        y=alt.Y("y:O", axis=None),
        text=alt.Text("tag:N")
    )
    .properties(width=240, height=32 * len(tag_order))  # Only here!
)

# Compose as side-by-side: labels + (lines + nodes)
lines = lines.properties(height=32 * len(tag_order))  # set height here
nodes = nodes.properties(height=32 * len(tag_order))  # set height here

# Do NOT set width on the HConcatChart itself!
chart = alt.hconcat(
    row_labels,
    (lines + nodes).properties(width=60 * max_chain_length)
).resolve_legend(color="independent").properties(
    title="Full dependency tag chains (VO3 root → Curiosity leaf) — all chain steps shown"
)

chart

alt.HConcatChart(...)